> **Exploratory notebook (spring 2025), kept for reference.** It is superseded by the `marche_stats` package and `docs/results.md`.
> Known issues fixed in the package: the TOTAL rows are mixed with sector rows (double counting), forecast accuracy is measured in-sample instead of on a hold-out period, the seasonal period is set to 6 instead of 12, and municipalities transferred to Emilia-Romagna are not removed.


# Marche Companies Project


In [ ]:
import pandas as pd

# Load the CSV file
file_path = '../data/raw/active_companies_marche.csv'
data = pd.read_csv(file_path, delimiter=';')

# Print the first 10 rows
data.head(10)

In [ ]:
total_nan = data.isna().sum().sum()
print(f"Total number of NaN values: {total_nan}")

In [ ]:
pd.isna(data).sum()
# Check for duplicates
duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Project Ideas

1. **Dimensionality Reduction and Visualization**:
    - Reducing data dimensions and visualizing trends is relatively straightforward using tools like PCA or t-SNE.

2. **Trend Analysis**:
    - Identifying increasing, decreasing, or stable trends can be done using simple regression models or statistical methods.

3. **Clustering**:
    - Grouping territories or sectors based on trends is moderately easy using algorithms like K-Means.

4. **Anomaly Detection**:
    - Detecting outliers requires some understanding of anomaly detection algorithms like Isolation Forest.

5. **Classification**:
    - Predicting the sector of a territory involves more complexity, as it requires labeled data and training classification models.

6. **Churn Prediction**:
    - Predicting significant drops in active companies involves binary classification and feature engineering.

7. **Time-Series Forecasting**:
    - Forecasting future values is the most challenging, as it requires advanced models like ARIMA, Prophet, or LSTM and a good understanding of time-series data.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Extract the date columns (all columns except the first two)
date_columns = data.columns[2:-2]
date_data = data[date_columns]

# Perform PCA to reduce dimensions to 2
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(date_data)

# Add the reduced dimensions to the dataframe
data['PCA1'] = reduced_data[:, 0]
data['PCA2'] = reduced_data[:, 1]

# Plot the reduced data
plt.figure(figsize=(10, 6))
plt.scatter(data['PCA1'], data['PCA2'], alpha=0.5)
plt.title('Dimensionality Reduction using PCA')
plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.show()

This code performs **Principal Component Analysis (PCA)** to reduce the dimensionality of the dataset and visualize it in two dimensions. Here's a breakdown of what the code does:

1. **Extract Date Columns**:
    - The `date_columns` variable contains all columns representing dates (excluding the first two columns, which are categorical).
    - The `date_data` DataFrame contains only the numerical data corresponding to these date columns.

2. **Perform PCA**:
    - PCA reduces the high-dimensional data (192 date columns) into two principal components (`PCA1` and `PCA2`).
    - These components capture the maximum variance in the data, meaning they represent the most significant patterns or trends.

3. **Add PCA Results to the DataFrame**:
    - The reduced dimensions (`PCA1` and `PCA2`) are added as new columns to the `data` DataFrame.

4. **Plot the Reduced Data**:
    - A scatter plot is created using `PCA1` and `PCA2` to visualize the data in two dimensions.

### How to Interpret the Results:
- **Clusters**:
  - Points that are close together in the scatter plot represent rows in the original dataset that have similar trends or patterns across the date columns. 
  - If distinct clusters appear, it may indicate groups of territories or sectors with similar behaviors.

- **Outliers**:
  - Points far away from the main cluster may represent outliers or unusual trends in the data.

- **Axes (PCA1 and PCA2)**:
  - These axes are linear combinations of the original features (date columns) and represent the directions of maximum variance in the data.
  - PCA1 captures the most variance, while PCA2 captures the second most variance.

This visualization helps to identify patterns, clusters, or anomalies in the data that may not be apparent in the high-dimensional space.

In [ ]:
import matplotlib.pyplot as plt

unique_territories = data['Territorio'].nunique()
print(f"Number of unique territories: {unique_territories}")

unique_sectors = data['Settore Ateco 2007'].nunique()
print(f"Number of unique sectors: {unique_sectors}")

## Clustering

In [ ]:
from sklearn.cluster import KMeans

# Determine the optimal number of clusters using the elbow method
inertia = []
k_range = range(2, 10)  # Testing different cluster numbers

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data[['PCA1', 'PCA2']])
    inertia.append(kmeans.inertia_)

# Plot the Elbow Method
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

In [ ]:
optimal_k = 5 

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
data['Cluster'] = kmeans.fit_predict(data[['PCA1', 'PCA2']])

# Visualize clusters
plt.figure(figsize=(10, 6))
plt.scatter(data['PCA1'], data['PCA2'], c=data['Cluster'], cmap="viridis", alpha=0.5)
plt.title('K-Means Clustering of Time-Series Trends')
plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.colorbar(label="Cluster")
plt.show()

In [ ]:
data.groupby("Cluster")["Territorio"].value_counts()

In [ ]:
data.groupby("Cluster")["Settore Ateco 2007"].value_counts()

In [ ]:
data.head(10)

In [ ]:
# Add province column
data['Provincia'] = data['Territorio'].str.split('-', expand=True)[0]
data['Provincia'].unique()

In [ ]:
# Sostituisci 'PS ' con 'PU' e 'AP ' con 'AP'
data['Provincia'] = data['Provincia'].str.replace('PS ', 'PU', regex=False)
data['Provincia'] = data['Provincia'].str.replace('AP ', 'AP', regex=False)

# Verifica le modifiche
print(data['Provincia'].unique())

Costa o entroterra?
Considera provincia
Considera settore

## Trend Analysis

In [ ]:
# Adjust this slice as needed based on your DataFrame structure.
date_columns = data.columns[2:-2]  # assuming last two columns are PCA1 and PCA2 or Cluster

# Melt the data into a long format
melted = pd.melt(data, 
                 id_vars=["Territorio", "Settore Ateco 2007", "PCA1", "PCA2", "Cluster", "Provincia"],
                 value_vars=date_columns,
                 var_name='Date',
                 value_name='ActiveCompanies')

# Convert date strings to datetime objects (adjust format if needed)
melted['Date'] = pd.to_datetime(melted['Date'], dayfirst=True)

# Sort the data by date
melted.sort_values('Date', inplace=True)

In [ ]:
# Group by Cluster and Date, then calculate the mean number of active companies
cluster_trends = melted.groupby(["Cluster", "Date"])["ActiveCompanies"].mean().reset_index()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.lineplot(data=cluster_trends, x="Date", y="ActiveCompanies", hue="Cluster", marker="o")
plt.title("Average Active Companies Over Time by Cluster")
plt.xlabel("Date")
plt.ylabel("Average Active Companies")
plt.legend(title="Cluster")
plt.show()


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Loop through each unique cluster
clusters = cluster_trends["Cluster"].unique()

for cluster in clusters:
    cluster_data = cluster_trends[cluster_trends["Cluster"] == cluster].copy()
    
    # Convert dates to numerical format (ordinal values)
    # This gives a numeric representation of each date for regression
    cluster_data['DateNumeric'] = cluster_data['Date'].map(pd.Timestamp.toordinal)
    
    # Prepare X and y for regression
    X = cluster_data['DateNumeric'].values.reshape(-1, 1)
    y = cluster_data['ActiveCompanies'].values
    
    # Fit linear regression
    reg = LinearRegression()
    reg.fit(X, y)
    
    # Calculate predicted trend
    cluster_data['Trend'] = reg.predict(X)
    slope = reg.coef_[0]
    intercept = reg.intercept_
    
    # Plot the original time series and the trend line
    plt.figure(figsize=(10, 5))
    plt.plot(cluster_data['Date'], cluster_data['ActiveCompanies'], marker='o', label="Average Active Companies")
    plt.plot(cluster_data['Date'], cluster_data['Trend'], label="Trend Line", linewidth=2)
    plt.title(f"Cluster {cluster} Trend Analysis\nSlope: {slope:.2f}")
    plt.xlabel("Date")
    plt.ylabel("Average Active Companies")
    plt.legend()
    plt.show()
    
    print(f"Cluster {cluster}: Slope = {slope:.2f} (Intercept = {intercept:.2f})")

### Introducing Seasonality

Step 1: Check for Seasonality

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Select one cluster to analyze first
cluster = 4  # Change this to analyze different clusters
cluster_data = cluster_trends[cluster_trends["Cluster"] == cluster].copy()

# Set Date as index
cluster_data.set_index("Date", inplace=True)

# Decompose time series
result = seasonal_decompose(cluster_data["ActiveCompanies"], model='additive', period=12)  # Adjust period based on your data

# Plot the decomposition
fig, axes = plt.subplots(4, 1, figsize=(10, 8))
result.observed.plot(ax=axes[0], title='Observed')
result.trend.plot(ax=axes[1], title='Trend')
result.seasonal.plot(ax=axes[2], title='Seasonality')
result.resid.plot(ax=axes[3], title='Residuals')
plt.tight_layout()
plt.show()


Step 2: Apply SARIMA for Forecasting

SARIMA is an extension of ARIMA that captures seasonality. It has three seasonal parameters (P, D, Q, S), where S is the seasonal period.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Define seasonal parameters (you may need to adjust based on your decomposition)
seasonal_order = (1, 1, 1, 6)  # (P, D, Q, S) where S is the seasonal period

# Fit SARIMA model
sarima_model = SARIMAX(cluster_data["ActiveCompanies"], order=(1,1,1), seasonal_order=seasonal_order)
sarima_result = sarima_model.fit()

# Forecast next 12 months
forecast_steps = 12
future_dates = pd.date_range(start=cluster_data.index[-1], periods=forecast_steps+1, freq='M')[1:]

# Generate predictions
sarima_forecast = sarima_result.get_forecast(steps=forecast_steps)
forecast_mean = sarima_forecast.predicted_mean

# Plot actual vs predicted
plt.figure(figsize=(10, 5))
plt.plot(cluster_data.index, cluster_data["ActiveCompanies"], label="Observed", marker="o")
plt.plot(future_dates, forecast_mean, label="Forecast", linestyle="dashed", marker="o", color="red")
plt.fill_between(future_dates, sarima_forecast.conf_int()['lower ActiveCompanies'], 
                 sarima_forecast.conf_int()['upper ActiveCompanies'], alpha=0.2, color="red")
plt.title(f"SARIMA Forecast for Cluster {cluster}")
plt.xlabel("Date")
plt.ylabel("Active Companies")
plt.legend()
plt.show()


Step 3: Apply Facebook Prophet for an Alternative Forecast
Prophet is a great model by Facebook that automatically detects seasonality, trend shifts, and outliers.

In [ ]:
from prophet import Prophet

# Prepare Data for Prophet (rename columns)
prophet_data = cluster_data.reset_index().rename(columns={"Date": "ds", "ActiveCompanies": "y"})

# Initialize and fit the model
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
prophet_model.fit(prophet_data)

# Generate future dates (next 12 months)
future_dates = prophet_model.make_future_dataframe(periods=12, freq='M')

# Make Predictions
forecast = prophet_model.predict(future_dates)

# Plot Forecast
fig = prophet_model.plot(forecast)
plt.title(f"Facebook Prophet Forecast for Cluster {cluster}")
plt.show()

# Plot Components (Trend and Seasonality)
fig2 = prophet_model.plot_components(forecast)
plt.show()

Step 5: Evaluate the Forecasts
To compare SARIMA and Prophet, measure how well they fit historical data using a metric like Mean Absolute Error (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error

# Get in-sample predictions for SARIMA
sarima_preds = sarima_result.get_prediction(start=cluster_data.index[0], end=cluster_data.index[-1])
sarima_mae = mean_absolute_error(cluster_data["ActiveCompanies"], sarima_preds.predicted_mean)

# Get in-sample predictions for Prophet
prophet_forecast = forecast.set_index("ds").loc[cluster_data.index]["yhat"]
prophet_mae = mean_absolute_error(cluster_data["ActiveCompanies"], prophet_forecast)

print(f"SARIMA MAE: {sarima_mae:.2f}")
print(f"Prophet MAE: {prophet_mae:.2f}")


# Step 6: Apply to All Clusters
Once you're comfortable with SARIMA and Prophet, loop through all clusters and generate forecasts.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Loop through each unique cluster for SARIMA and Prophet
for cluster in clusters:
    cluster_data = cluster_trends[cluster_trends["Cluster"] == cluster].copy()

    # Ensure Date is a datetime index
    cluster_data["Date"] = pd.to_datetime(cluster_data["Date"])
    cluster_data = cluster_data.set_index("Date").dropna()

    '''
    SARIMA Forecasting
    '''
    seasonal_order = (1, 1, 1, 6)  # Adjust if needed

    # Fit SARIMA model
    sarima_model = SARIMAX(cluster_data["ActiveCompanies"], order=(1,1,1), seasonal_order=seasonal_order)
    sarima_result = sarima_model.fit()

    # Forecast next 12 months
    forecast_steps = 12
    future_dates = pd.date_range(start=cluster_data.index[-1], periods=forecast_steps+1, freq='M')[1:]

    # Generate predictions
    sarima_forecast = sarima_result.get_forecast(steps=forecast_steps)
    forecast_mean = sarima_forecast.predicted_mean

    # Plot actual vs predicted
    plt.figure(figsize=(10, 5))
    plt.plot(cluster_data.index, cluster_data["ActiveCompanies"], label="Observed", marker="o")
    plt.plot(future_dates, forecast_mean, label="Forecast", linestyle="dashed", marker="o", color="red")
    plt.fill_between(future_dates, sarima_forecast.conf_int().iloc[:, 0], 
                    sarima_forecast.conf_int().iloc[:, 1], alpha=0.2, color="red")
    plt.title(f"SARIMA Forecast for Cluster {cluster}")
    plt.xlabel("Date")
    plt.ylabel("Active Companies")
    plt.legend()
    plt.show()
    
    # Get in-sample predictions for SARIMA
    sarima_preds = sarima_result.get_prediction(start=cluster_data.index[0], end=cluster_data.index[-1], dynamic=False)
    sarima_mae = mean_absolute_error(cluster_data["ActiveCompanies"], sarima_preds.predicted_mean)

    '''
    Prophet Forecasting
    '''
    # Prepare Data for Prophet
    prophet_data = cluster_data.reset_index().rename(columns={"Date": "ds", "ActiveCompanies": "y"})

    # Initialize and fit the model
    prophet_model = Prophet(yearly_seasonality=True)
    prophet_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    prophet_model.fit(prophet_data)

    # Generate future dates (next 12 months)
    future_dates = prophet_model.make_future_dataframe(periods=12, freq='M')

    # Make Predictions
    forecast = prophet_model.predict(future_dates)

    # Plot Forecast
    fig = prophet_model.plot(forecast)
    plt.title(f"Facebook Prophet Forecast for Cluster {cluster}")
    plt.show()

    # Plot Components (Trend and Seasonality)
    fig2 = prophet_model.plot_components(forecast)
    plt.show()

    # Get in-sample predictions for Prophet
    prophet_forecast = forecast.set_index("ds").reindex(cluster_data.index)["yhat"]
    prophet_mae = mean_absolute_error(cluster_data["ActiveCompanies"], prophet_forecast)

    print(f"Cluster: {cluster} - SARIMA MAE: {sarima_mae:.2f}")
    print(f"Cluster: {cluster} - Prophet MAE: {prophet_mae:.2f}")

In [ ]:
# Let's work with Prophet now
from prophet import Prophet
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

prophet_mae_dic = {}
# Loop through each unique cluster for SARIMA and Prophet
for cluster in clusters:
    cluster_data = cluster_trends[cluster_trends["Cluster"] == cluster].copy()

    # Ensure Date is a datetime index
    cluster_data["Date"] = pd.to_datetime(cluster_data["Date"])
    cluster_data = cluster_data.set_index("Date").dropna()

    '''
    Prophet Forecasting
    '''
    # Prepare Data for Prophet
    prophet_data = cluster_data.reset_index().rename(columns={"Date": "ds", "ActiveCompanies": "y"})

    if cluster == 4:
        # Initialize and fit the model
        prophet_model = Prophet(changepoint_prior_scale=0.3, yearly_seasonality=True)
        prophet_model.add_seasonality(name='quarterly', period=91.25, fourier_order=8)
    elif cluster == 1:
        # Initialize and fit the model
        prophet_model = Prophet(changepoint_prior_scale=0.3, yearly_seasonality=True)
        prophet_model.add_seasonality(name='quarterly', period=91.25, fourier_order=8)
    else:
        prophet_model = Prophet(yearly_seasonality=True)
        prophet_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
           
    prophet_model.fit(prophet_data)

    # Generate future dates (next 12 months)
    future_dates = prophet_model.make_future_dataframe(periods=12, freq='M')

    # Make Predictions
    forecast = prophet_model.predict(future_dates)

    '''# Plot Forecast
    fig = prophet_model.plot(forecast)
    plt.title(f"Facebook Prophet Forecast for Cluster {cluster}")
    plt.show()

    # Plot Components (Trend and Seasonality)
    fig2 = prophet_model.plot_components(forecast)
    plt.show()'''

    # Get in-sample predictions for Prophet
    prophet_forecast = forecast.set_index("ds").reindex(cluster_data.index)["yhat"]
    prophet_mae = mean_absolute_error(cluster_data["ActiveCompanies"], prophet_forecast)
    
    # Store the MAE for each cluster
    prophet_mae_dic[cluster] = prophet_mae
    
# Display the MAE for each cluster
print("Prophet MAE for each cluster:")
for cluster, mae in prophet_mae_dic.items():
    print(f"Cluster {cluster}: MAE = {mae:.2f}")


4️⃣ Consider Switching Models
If Prophet still underperforms, you might want to try a simple regression model like XGBoost or SARIMA instead.

Prophet works best with a lot of data, so for tiny datasets, classical models may work better.

# **Provincial Analysis**
    - **Goal**: Analyze trends and patterns at the province level.
    - **Steps**:
      - Group the data by `Provincia` and calculate metrics like the average number of active companies, growth rates, or trends over time.
      - Visualize the data using bar charts, line plots, or heatmaps to compare provinces.

In [ ]:
import seaborn as sns
import geopandas as gpd
import folium

province_trends = melted.groupby(["Provincia", "Date"])["ActiveCompanies"].mean().reset_index()

# Plot trends for each province
plt.figure(figsize=(12, 6))
sns.lineplot(data=province_trends, x="Date", y="ActiveCompanies", hue="Provincia", marker="o")
plt.title("Active Companies Over Time by Province")
plt.xlabel("Date")
plt.ylabel("Average Active Companies")
plt.legend(title="Province", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### 2. **Province-Level Forecasting**
    - **Goal**: Forecast the number of active companies for each province.
    - **Steps**:
      - Use models like SARIMA or Prophet to forecast future trends for each province.
      - Compare forecasts across provinces to identify regions with potential growth or decline.

In [ ]:
from prophet import Prophet

province_trends = melted.groupby(["Provincia", "Date"])["ActiveCompanies"].sum().reset_index()

# Initialize a dictionary to store forecasts for each province
province_forecasts = {}

# Initialize a DataFrame to store all forecasts for plotting
all_forecasts = pd.DataFrame()

# Loop through each unique province
for province in province_trends["Provincia"].unique():
    # Filter data for the current province
    province_data = province_trends[province_trends["Provincia"] == province].copy()
    
    # Prepare data for Prophet
    province_data.rename(columns={"Date": "ds", "ActiveCompanies": "y"}, inplace=True)
    
    # Initialize and fit the Prophet model
    prophet_model = Prophet(yearly_seasonality=True)
    prophet_model.fit(province_data)
    
    # Generate future dates (next 12 months)
    future_dates = prophet_model.make_future_dataframe(periods=12, freq='M')
    
    # Make predictions
    forecast = prophet_model.predict(future_dates)
    
    # Store the forecast in the dictionary
    province_forecasts[province] = forecast
    
    # Add the forecast to the all_forecasts DataFrame
    forecast["Provincia"] = province
    all_forecasts = pd.concat([all_forecasts, forecast[["ds", "yhat", "Provincia"]]])

# Add a column to indicate whether the data is forecasted or historical
all_forecasts["Forecasted"] = all_forecasts["ds"] > province_trends["Date"].max()

# Plot all forecasts with historical and forecasted data highlighted
plt.figure(figsize=(14, 8))
sns.lineplot(data=all_forecasts, x="ds", y="yhat", hue="Provincia", style="Forecasted", markers=True, palette="tab10")
plt.title("Forecasted Active Companies by Province")
plt.xlabel("Date")
plt.ylabel("Forecasted Active Companies")
plt.legend(title="Province", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


### 4. **Province Comparison**
    - **Goal**: Compare provinces based on key metrics like growth rate, seasonality, or anomalies.
    - **Steps**:
      - Calculate growth rates or seasonal patterns for each province.
      - Highlight provinces with the highest/lowest growth or unusual trends.

In [ ]:
import pandas as pd
import numpy as np

# Ensure the "Date" column is treated as a datetime object
melted["Date"] = pd.to_datetime(melted["Date"])

# Step 1: Aggregate the sum of active companies per province per date
aggregated = (
    melted.groupby(["Provincia", "Date"])["ActiveCompanies"]
    .sum()
    .reset_index()  # Reset index to return a DataFrame
)

# Step 2: Sort by province and date to ensure proper calculation of growth rates
aggregated = aggregated.sort_values(by=["Provincia", "Date"])

# Step 3: Calculate the daily growth rate of active companies per province
aggregated["GrowthRate"] = (
    aggregated.groupby("Provincia")["ActiveCompanies"]
    .pct_change()  # Calculate percentage change from one day to the next
    .replace([np.inf, -np.inf], np.nan)  # Replace infinite values with NaN
    .fillna(0)  # Fill NaN values with 0
)

# Step 4: Sort provinces by the least negative average growth rates (closest to zero)
top_provinces = (
    aggregated.groupby("Provincia")["GrowthRate"]
    .mean()
    .sort_values(ascending=False)  # Sort by least negative growth rates
)

print("Top Provinces with Least Decline in Growth Rate:")
print(top_provinces)

In [ ]:
import pandas as pd
import numpy as np

# Ensure the "Date" column is treated as a datetime object
melted["Date"] = pd.to_datetime(melted["Date"])

# Step 1: Aggregate the sum of active companies per province per date
aggregated = (
    melted.groupby(["Provincia", "Date"])["ActiveCompanies"]
    .sum()
    .reset_index()  # Reset index to return a DataFrame
)

# Step 2: Sort by province and date to ensure proper ordering
aggregated = aggregated.sort_values(by=["Provincia", "Date"])

# Step 3: Calculate the absolute change (last value - first value) for each province
absolute_change = (
    aggregated.groupby("Provincia")["ActiveCompanies"]
    .agg(["first", "last"])  # Get the first and last values for each province
    .reset_index()  # Reset index to return a DataFrame
)

# Add a new column for the absolute change
absolute_change["AbsoluteChange"] = absolute_change["last"] - absolute_change["first"]

# Step 4: Sort provinces by the absolute change
absolute_change = absolute_change.sort_values(by="AbsoluteChange", ascending=False)

print("Provinces Sorted by Absolute Change in Active Companies:")
print(absolute_change)

### 6. **Sector Analysis by Province**
    - **Goal**: Analyze how different sectors perform across provinces.
    - **Steps**:
      - Group data by `Provincia` and `Settore Ateco 2007` to identify dominant sectors in each province.
      - Visualize sector distribution using stacked bar charts or pie charts.

    ```python
    sector_province = melted.groupby(["Provincia", "Settore Ateco 2007"])["ActiveCompanies"].sum().unstack()

    # Plot sector distribution for a specific province
    province = "AN"  # Replace with desired province
    sector_province.loc[province].plot(kind="bar", figsize=(10, 6))
    plt.title(f"Sector Distribution in Province: {province}")
    plt.xlabel("Sector")
    plt.ylabel("Total Active Companies")
    plt.show()
    ```

---

### 7. **Policy Recommendations**
    - **Goal**: Use insights to recommend policies for economic development.
    - **Steps**:
      - Identify provinces or sectors with declining trends and suggest interventions.
      - Highlight high-growth provinces or sectors for investment opportunities.
